In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_config

In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_helpers

## 1. Read from Bronze

In [0]:
log("Reading from Bronze ...")
df_bronze = spark.table(TBL_BRONZE_RAW)
display(df_bronze.limit(10))


## 2. Extract Customer Columns & Add Ingestion Metadata

In [0]:
df_customers = df_bronze.select(
    F.col("customer_id"),
    F.col("customer_name"),
    F.col("segment"),
    F.col("ingested_at"),
    F.col("file_path"),
    F.col("file_name"),
    F.col("file_size")
    ) \
    .withColumn("transformed_at", F.current_timestamp()) 
display(df_customers)

## 3. Deduplicate

In [0]:
df_duplicate = df_customers.groupBy("customer_id").count().where("count > 1")
df_duplicate.show()

In [0]:
log(f"Rows before duplicate drop: {df_customers.count():,}")
df_silver_customers = df_customers.dropDuplicates(["customer_id"])
log(f"Rows after duplicate drop: {df_silver_customers.count():,}")

## 4. Standardize Segment Values

In [0]:
df_silver_customers = df_silver_customers. \
    withColumn("segment",F.initcap(F.trim(F.col("segment")))
    )

display(df_silver_customers)

## 5. Write to Silver

In [0]:
log(f"Writing to {TBL_SILVER_CUSTOMERS} [mode=overwrite] ...")
df_silver_customers.write\
    .format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(TBL_SILVER_CUSTOMERS)

count = get_row_count(TBL_SILVER_CUSTOMERS)
log(f"✅ Done. {count:,} rows written to {TBL_SILVER_CUSTOMERS}")

In [0]:
display(spark.table(TBL_SILVER_CUSTOMERS).limit(10))